# 04 — Evaluation

Evaluation of survival-model discrimination, calibration and model explanations, with subgroup analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sksurv.metrics import concordance_index_censored

## Evaluation metrics

The study protocol specifies Uno's C-index, time-dependent AUROC, Brier/Integrated Brier Score, calibration and clinical utility measures. The exact implementation should be applied to the held-out test set.

In [ ]:
results = pd.DataFrame({
    'model': ['Elastic-Net Cox', 'Random Survival Forest'],
    'test_c_index': [cox_test_cindex, rsf_test_cindex],
})

results

## Risk-score distributions

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(cox_test_pred, bins=30)
plt.xlabel('Cox risk score')
plt.ylabel('Patients')
plt.title('Cox Test Risk Scores')
plt.tight_layout()
plt.show()

## Cox coefficients

In [ ]:
alpha_index = np.argmin(np.abs(cox.alphas_ - best_alpha))
coefficients = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': cox.coef_[:, alpha_index],
})
coefficients['abs_coefficient'] = coefficients['coefficient'].abs()
coefficients.sort_values('abs_coefficient', ascending=False)

## SHAP / model explanation

In [ ]:
import shap

background = X_train_scaled[:100]
explain = X_test_scaled[:100]

predict_fn = lambda x: cox.predict(x, alpha=best_alpha)
explainer = shap.Explainer(predict_fn, background)
shap_values = explainer(explain)

shap.summary_plot(
    shap_values,
    explain,
    feature_names=feature_cols,
    show=True,
)

## Random Survival Forest feature importance

The scikit-survival Random Survival Forest implementation does not expose the standard sklearn `feature_importances_` property. Permutation importance can therefore be used as a model-agnostic feature-importance analysis.

In [ ]:
from sklearn.base import BaseEstimator

class RSFEstimator:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return self.model.predict(X)

    def score(self, X, y):
        prediction = self.model.predict(X)
        return concordance_index_censored(
            y['event'], y['time'], prediction
        )[0]

## Subgroup analysis

In [ ]:
test_df = df.loc[test_idx].copy()
test_df['cox_risk_score'] = cox_test_pred
test_df['rsf_risk_score'] = rsf_test_pred

if 'age' in test_df:
    test_df['age_group'] = np.where(
        test_df['age'] >= 65,
        '>=65',
        '<65'
    )

for group, group_df in test_df.groupby('age_group'):
    if group_df['event'].sum() > 0:
        cindex = concordance_index_censored(
            group_df['event'].astype(bool),
            group_df['time'].astype(float),
            group_df['cox_risk_score'],
        )[0]
        print(group, 'C-index:', round(cindex, 4))

## Final interpretation

Performance should be interpreted only on the held-out test set. Synthetic results demonstrate software/model behavior and do not establish clinical validity.